In [1]:
import os

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

/tmp/ipykernel_14590/1788999079.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
# all relevant health & wellness documents
documents = [
    Document(page_content="Regular exercise is essential for maintaining good health and wellness.", metadata={"source": "health_article_1"}),
    Document(page_content="A balanced diet that includes a variety of fruits and vegetables is crucial for overall well-being.", metadata={"source": "health_article_2"}),
    Document(page_content="Getting enough sleep is vital for physical and mental health.", metadata={"source": "health_article_3"}),
    Document(page_content="Stress management techniques, such as meditation and deep breathing, can improve health and wellness.", metadata={"source": "health_article_4"}),
    Document(page_content="Regular health check-ups and screenings are important for early detection and prevention of diseases.", metadata={"source": "health_article_5"}),
    Document(page_content="Python balances readibility and performance, making it a popular choice for developers.", metadata={"source": "tech_article_1"}),
    Document(page_content="JavaScript is a versatile language used for both front-end and back-end web development.", metadata={"source": "tech_article_2"}),
    Document(page_content="Photosynthesis is the process by which plants convert sunlight into energy, producing oxygen as a byproduct.", metadata={"source": "science_article_1"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar, marking the first time the tournament took place in the Middle East.", metadata={"source": "sports_article_1"}),
    Document(page_content="Black holes are regions in space where gravity is so strong that nothing, not even light, can escape from them.", metadata={"source": "science_article_2"}),
    Document(page_content="The Great Wall of China is a series of fortifications built to protect against invasions and raids from various nomadic groups.", metadata={"source": "history_article_1"})
]

In [9]:
# Initialize the OpenaAI Embeddings model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=30,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)


In [10]:
# Create a FAISS vector store from the documents
vector_store = FAISS.from_documents(documents, embedding_model)

In [11]:
# create  retriever
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [12]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOpenAI(
        model="openai/gpt-4o-mini",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1",
        temperature=0,
    ),
)

In [13]:
# Query
query = "How to improve energy levels and reduce fatigue?"

In [14]:
# Retrieve results

similarty_result = similarity_retriever.invoke(query)
multi_query_result = multi_query_retriever.invoke(query)


In [15]:
for i, result in enumerate(similarty_result):
    print(f"Similarity Result {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Source: {result.metadata['source']}")
    print()

print("--------------------------------------------------\n")

for i, result in enumerate(multi_query_result):
    print(f"Multi-Query Result {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Source: {result.metadata['source']}")
    print()

Similarity Result 1:
Content: Photosynthesis is the process by which plants convert sunlight into energy, producing oxygen as a byproduct.
Source: science_article_1

Similarity Result 2:
Content: Stress management techniques, such as meditation and deep breathing, can improve health and wellness.
Source: health_article_4

Similarity Result 3:
Content: Regular health check-ups and screenings are important for early detection and prevention of diseases.
Source: health_article_5

Similarity Result 4:
Content: Regular exercise is essential for maintaining good health and wellness.
Source: health_article_1

Similarity Result 5:
Content: Getting enough sleep is vital for physical and mental health.
Source: health_article_3

--------------------------------------------------

Multi-Query Result 1:
Content: Photosynthesis is the process by which plants convert sunlight into energy, producing oxygen as a byproduct.
Source: science_article_1

Multi-Query Result 2:
Content: Regular exercise is es